# Task 8: Long Softmax

Bot task id: `08_long_softmax`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/08_long_softmax.ipynb)

## Условие

Построчный softmax для матрицы `x` формы `(N0, T)`:

$$\text{row\_max}_i = \max_j x[i, j], \qquad out[i, j] = \frac{e^{x[i, j] - \text{row\_max}_i}}{\sum_k e^{x[i, k] - \text{row\_max}_i}}$$

**Требования**
- одномерная сетка: `pid0` перебирает строки `i`;
- тайлы длины `B1 < T` и маски для хвостов;
- первый проход — `row_max`;
- затем в цикле — сумма экспонент в стабильной форме;
- последний проход — нормализовать и сохранить;
- `exp2(x * log2(e))` вместо `exp(x)`;
- решение должно работать на GPU (CUDA).

Проверка: `x` формы `(4, 200)`, значения порядка ±15, так что без вычитания максимума точность пострадает.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `08_long_softmax.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 08_long_softmax.py
import torch
import triton
import triton.language as tl


@triton.jit
def _long_softmax_kernel(x_ptr, out_ptr, N0, T, B1: tl.constexpr):
    # YOUR CODE HERE
    pass


def softmax_triton(x: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "08_long_softmax.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x = torch.randn(4, 200, device="cuda") * 5
torch.testing.assert_close(sol.softmax_triton(x), torch.softmax(x, 1), atol=1e-5, rtol=1e-4)
assert "exp2" in SRC, "the task asks for exp2(x * log2(e))"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("08_long_softmax.py")